# Segment a survey-scale LiDAR tile

A survey arrives as tiles rather than as scenes: tens of millions of points per file, one file per
city block or map sheet, and you return one label per point, on the original points, in the original order. None of that fits in a forward pass.

This notebook is the production-inference path:

1. **The machinery**, on a synthetic survey tile with a toy predictor. It runs anywhere, with no
   model, no weights and no dataset.
2. **A real 21 M point mobile-mapping tile**, segmented at full point resolution by a pretrained
   checkpoint. That half needs the dataset, the sparse-convolution extra and downloaded weights, so
   it is guarded by a path check.

New to running a model over a scene? Start with [Segment a scene](02-segmentation-inference.md).

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import time
from pathlib import Path

import torch
import torch.nn.functional as F

import torch_pointcloud as tp
import torch_pointcloud.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

In [ ]:
import matplotlib.pyplot as plt

DRAWN = 60_000  # scattering 21 M points takes minutes and comes out as one opaque blob
MAP, STREET = (78, 0), (58, -78)  # elevation and azimuth: the tile as a map, and one section of street


def show_cloud(pos, color=None, *, ax=None, title=None, size=1.0, view=MAP, cmap="tab20", vmin=None, vmax=None):
    """Scatter a cloud, subsampled to DRAWN points. `pos` is (N, 3); `color` is a label vector or None.

    Pass `vmin` and `vmax` whenever two panels show label vectors that must agree: without them
    matplotlib normalizes each panel against its own range and the same class comes out two colors.
    """
    if ax is None:
        ax = plt.figure(figsize=(7, 4)).add_subplot(projection="3d")
    keep = torch.randperm(len(pos))[:DRAWN]
    p = pos[keep].cpu().numpy()
    c = color[keep].cpu().numpy() if torch.is_tensor(color) else color
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, cmap=cmap, vmin=vmin, vmax=vmax,
               depthshade=False, linewidths=0)
    ax.view_init(elev=view[0], azim=view[1])
    ax.set_box_aspect(p.max(0) - p.min(0))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## A tile is not a scene

An indoor scene is a closed room a few meters across. A survey tile is a corridor of street or a
square kilometer of terrain, sampled at a density set by the vehicle's speed and the scanner's
rotation rate, not by anyone's memory budget.

The synthetic tile below is built to that shape: $220$ m of street, two facades, street trees,
lamp posts and parked cars, at a density in the range a mobile mapping run produces.

In [ ]:
def make_survey_tile(seed=0):
    """A synthetic mobile-mapping tile: 220 m of street, two facades, street trees, poles, parked cars."""
    generator = torch.Generator().manual_seed(seed)

    def box(count, low, high):
        low, high = torch.tensor(low), torch.tensor(high)
        return torch.rand(count, 3, generator=generator) * (high - low) + low

    road = box(1_100_000, [0.0, -11.0, -0.04], [220.0, 11.0, 0.04])
    facades = torch.cat([
        box(320_000, [0.0, -14.2, 0.0], [220.0, -13.9, 15.0]),
        box(320_000, [0.0, 13.9, 0.0], [220.0, 14.2, 15.0]),
    ])
    poles = torch.cat([box(1_600, [x, -9.6, 0.0], [x + 0.12, -9.48, 7.0]) for x in range(6, 215, 24)])
    trunks = torch.cat([box(2_000, [x, 8.7, 0.0], [x + 0.3, 9.0, 2.4]) for x in range(10, 215, 18)])
    crowns = torch.cat([box(24_000, [x - 2.0, 6.5, 2.4], [x + 2.5, 11.2, 7.5]) for x in range(10, 215, 18)])
    cars = torch.cat([box(9_000, [x, -10.2, 0.0], [x + 4.3, -8.4, 1.5]) for x in range(4, 215, 9)])

    pos = torch.cat([road, facades, poles, trunks, crowns, cars])
    pos[:, 2] += 0.004 * pos[:, 0]  # the street climbs gently, as a real one does
    return {"pos": pos, "batch": torch.zeros(len(pos), dtype=torch.long)}


tile = make_survey_tile()
pos = tile["pos"]
extent = (pos.amax(0) - pos.amin(0)).tolist()
footprint = extent[0] * extent[1]
print(f"points:    {len(pos):,}")
print(f"extent:    {extent[0]:.0f} x {extent[1]:.0f} x {extent[2]:.0f} m")
print(f"density:   {len(pos) / footprint:.0f} points per square meter over {footprint / 1e4:.2f} ha")
print(f"positions: {pos.numel() * pos.element_size() / 2**20:.0f} MiB as float32")

$2.28$ M points over $0.62$ ha, at $365$ points per square meter. The positions alone are $26$ MiB, but the positions are not what costs.

## Why one forward pass does not work

A voxel backbone does not consume raw points: it quantizes them to a grid and convolves the
occupied voxels. So the size that matters is **occupied voxels at the model's working resolution**,
and the memory that matters is the activations a network keeps alive over them.

In [ ]:
VOXEL = 0.05  # meters: the resolution the checkpoint used later was trained at

voxels = T.Voxelize(pos_key="pos", pos_reduce="grid", size=VOXEL)({"pos": pos})
count = voxels["pos"].shape[0]
print(f"{len(pos):,} points -> {count:,} occupied voxels at {VOXEL * 100:.0f} cm")
print(f"one float32 feature map at 32 channels: {count * 32 * 4 / 2**30:.2f} GiB")
print(f"a 20-map UNet, activations alone:       {20 * count * 32 * 4 / 2**30:.1f} GiB")

$2.1$ M voxels, $0.25$ GiB for a *single* 32-channel float32 feature map, and a sparse UNet keeps
dozens of them alive at once plus a kernel map per convolution. That is already several GiB for a
toy tile a tenth the size of a real one. The real tile below puts a measured number on that slope:
$4.63$ GiB per million voxels, which lands a single pass over its $7.16$ M voxels at $33$ GiB.

Shrinking the model does not help: the cost is linear in the tile, and tiles only get bigger. What you can change is **how much of the tile the model reads in one call**.

## Choosing the unit of work

That decision belongs to an [`Inferer`](../inferers/overview.md). It takes the packed
tile dict and a `predictor` callable, decides which sub-clouds the predictor sees, and stitches the
partial results back into one row per input point.

Three strategies matter at survey scale.

| Inferer | Unit of work | Choose it when |
| --- | --- | --- |
| [`SlidingWindowInferer`](../inferers/sliding-window.md) | axis-aligned blocks of a fixed **metric** size | the model was trained on scenes of a known physical size, and you want a deterministic tiling you can resume, checkpoint and parallelize |
| [`KNNWindowInferer`](../inferers/knn-window.md) | crops of a fixed **point budget** | density swings wildly across the tile, so metric blocks would hold anywhere from a thousand to a million points |
| [`VoxelPartitionInferer`](../inferers/voxel-partition.md) | whole-extent downsamples, one point per voxel | the model wants whole-scene context and the tile is compact enough that a downsampled pass still fits |

A survey tile is huge, roughly uniform in density along the trajectory, and needs to be resumable.
`SlidingWindowInferer` is the default; the rest of this notebook uses it.

To see the contract with no model at all, here is a toy predictor that labels points from their height and their distance to the
street axis.

In [ ]:
CLASSES = ["ground", "facade", "pole", "tree", "car"]


def toy_predictor(block):
    """Toy predictor: five 'classes' from height and distance to the street axis, as confident logits."""
    height, side = block["pos"][:, 2] - 0.004 * block["pos"][:, 0], block["pos"][:, 1].abs()
    label = torch.zeros(len(block["pos"]), dtype=torch.long)
    label[(height > 0.15) & (side > 12.0)] = 1
    label[(height > 0.15) & (side <= 12.0) & (height <= 2.2)] = 4
    label[(height > 2.2) & (side <= 12.0)] = 3
    label[(height > 2.2) & (side <= 12.0) & (block["pos"][:, 1] < 0.0)] = 2
    return F.one_hot(label, num_classes=5).float() * 6.0

In [ ]:
from torch_pointcloud.inferers import SlidingWindowInferer

started = time.perf_counter()
inferer = SlidingWindowInferer(block_size=40.0, overlap=0.25, mode="gaussian", dims=(0, 1))
probs = inferer(tile, predictor=toy_predictor)

print(f"output {tuple(probs.shape)} in {time.perf_counter() - started:.1f} s")
print("aligned to the input:", probs.shape[0] == len(pos))
labels, counts = probs.argmax(-1).unique(return_counts=True)
print({CLASSES[i]: f"{c / len(pos):.1%}" for i, c in zip(labels.tolist(), counts.tolist())})

show_cloud(pos, probs.argmax(-1), view=(62, -74), size=0.6, cmap="tab10",
           title="the synthetic tile, by the toy predictor's five classes");

`dims=(0, 1)` tiles in the ground plane only, so each block is a full-height column of the tile.
Cubic blocks would cut facades and tree crowns in half horizontally for no benefit: a survey tile is
wide and thin, and nothing about a street repeats vertically.

`overlap=0.25` with `mode="gaussian"` makes neighboring blocks share a quarter of their width and
weights each prediction by its distance to its own block center, so a point near a seam is decided
mostly by the block that saw it in context. At `overlap=0.0` the blocks partition the tile exactly
and each point is predicted once, which is faster and leaves visible seams.

### Sizing a block

From below, the block is bounded by the model's receptive field: this checkpoint's encoder
halves the resolution four times, so its deepest voxels are $2^4 \cdot 5\,\text{cm} = 0.8$ m
across and the context behind one prediction spans several meters. A block only a few meters wide
would be almost entirely edge.

From above, the survey's density, because points per block go as
$\text{density} \cdot \text{block}^2$. At the $394$ points per square meter of the real tile
below, a $40$ m block nominally holds $630$ k points and a $50$ m block $985$ k. Density is a
property of the survey and not of the model, so a block size that is comfortable on one survey can
be an out-of-memory error on the next at exactly the same metric size. Size the block in points,
then convert to meters through the density you actually have.

In [ ]:
for block_size in (20.0, 40.0, 80.0):
    seen = []

    def counting(block, seen=seen):
        seen.append(block["pos"].shape[0])
        return toy_predictor(block)

    started = time.perf_counter()
    SlidingWindowInferer(block_size=block_size, overlap=0.25, mode="gaussian", dims=(0, 1))(
        tile, predictor=counting
    )
    print(f"{block_size:>5.0f} m: {len(seen):>3} blocks, largest {max(seen):>9,} points, "
          f"{time.perf_counter() - started:.1f} s")

Doubling the block quarters the block count and quadruples the largest one: $30$ blocks of at most
$130$ k points, $8$ of at most $424$ k, $4$ of at most $845$ k. Pick the largest block whose worst
case still fits, then check that it did not cost accuracy. The real tile below measures both.

When a single block is still too large, `roi_num_points` caps the points per predictor call and
splits the block into sub-batches, predicting each point exactly once per block pass. The other
strategies batch the other way, packing several windows into one call: `sw_batch_size` on
`KNNWindowInferer`, `sub_batch_size` on `VoxelPartitionInferer`. Both are amortization knobs, and
neither changes what the model sees.

The other two strategies answer the same call, and the difference shows in how many times they call
the predictor for the same points.

In [ ]:
from torch_pointcloud.inferers import KNNWindowInferer, VoxelPartitionInferer

crop_idx = ((pos[:, 0] > 80.0) & (pos[:, 0] < 110.0)).nonzero().squeeze(1)
crop = {"pos": pos[crop_idx], "batch": torch.zeros(len(crop_idx), dtype=torch.long)}
print(f"crop: {len(crop_idx):,} points")

for name, strategy in [
    ("KNNWindowInferer", KNNWindowInferer(roi_num_points=32_768, overlap=0.5, aggregate="ema")),
    ("VoxelPartitionInferer", VoxelPartitionInferer(voxel_size=0.20, sub_batch_size=4)),
]:
    visits = [0]

    def counting(block, visits=visits):
        visits[0] += 1
        return toy_predictor(block)

    started = time.perf_counter()
    out = strategy(crop, predictor=counting)
    print(f"{name:22s} {visits[0]:>4} predictor calls, output {tuple(out.shape)}, "
          f"{time.perf_counter() - started:.1f} s")

## Preprocessing must be reversible

Every one of those strategies returns one row per input point, and that is the property the whole
workflow rests on. A voxel model breaks it in the middle: it predicts per *voxel*, and a voxel holds
three points on average in a mobile-mapping tile. The way back is the inverse map `Voxelize` records
under `dst_inverse_key`, which maps each original point to the voxel it fell into.

In [ ]:
voxelize = T.Voxelize(pos_key="pos", pos_reduce="grid", size=VOXEL, dst_inverse_key="inverse")
crop_voxels = voxelize({"pos": crop["pos"].clone()})
inverse = crop_voxels["inverse"]

print(f"{len(crop['pos']):,} points -> {crop_voxels['pos'].shape[0]:,} voxels")
print("inverse:", tuple(inverse.shape), inverse.dtype, "values in", (int(inverse.min()), int(inverse.max())))

voxel_logits = toy_predictor({"pos": crop_voxels["pos"].float() * VOXEL + crop["pos"].amin(0)})
full = voxel_logits[inverse]
print("per-voxel logits:", tuple(voxel_logits.shape), "-> per-point:", tuple(full.shape))
print("same length as the input:", full.shape[0] == crop["pos"].shape[0])

`inverse` has one entry per *original* point, so `voxel_logits[inverse]` is a gather back to full
resolution: same length, same order, no search and no nearest-neighbor step. Anything that changes
the row count downstream of the tile (voxelizing, padding, sampling) has to leave a map like this
behind, and `SlidingWindowInferer` knows how to use one: pass the per-block pipeline as `transform`
and tell it where the map lands with `inverse_key`.

Same length is easy to check. Same *order* deserves a test: permute the tile, run the same
inference, put the rows back, and compare.

In [ ]:
perm = torch.randperm(len(crop["pos"]))
shuffled = {"pos": crop["pos"][perm], "batch": crop["batch"]}

window = SlidingWindowInferer(block_size=10.0, overlap=0.25, mode="gaussian", dims=(0, 1))
straight = window(crop, predictor=toy_predictor)
scrambled = window(shuffled, predictor=toy_predictor)

restored = torch.empty_like(scrambled)
restored[perm] = scrambled
print("same shape:", straight.shape == restored.shape, tuple(straight.shape))
print("row for row identical:", bool(torch.allclose(straight, restored, atol=1e-6)))
print("largest difference:", float((straight - restored).abs().max()))

Identical to the last bit, on every one of the $317\,966$ rows. Block membership is decided by
position, so permuting the input permutes the output and nothing else; the inferer never reorders anything.

Two habits keep that guarantee useful. Never sort, shuffle or deduplicate the tile in place, because
the file you write back is indexed by the row order you read. And check length and order at the end
rather than trusting them, which is the last cell of this notebook.

## A real survey tile

The rest of the notebook runs a pretrained checkpoint over one tile of
[Paris-Lille-3D](https://npm3d.fr/paris-lille-3d), a mobile-mapping survey of French streets. The
cells need the raw scans on disk, the `torchsparse` extra and the downloaded weights.

> `Lille2.ply` is the held-out tile of the 10-class benchmark. Set `TORCH_POINTCLOUD_DATA_DIR` and
> place the scans under `$TORCH_POINTCLOUD_DATA_DIR/ParisLille3D/raw/`; the dataset requires
> accepting a license, so `torch-pointcloud` does not download it for you.

In [ ]:
from torch_pointcloud.config import DATA_DIR
from torch_pointcloud.datasets.parislille3d import PARISLILLE3D_CLASSES, load_parislille3d_data

TILE_PATH = Path(DATA_DIR) / "ParisLille3D" / "raw" / "Lille2.ply"
print(TILE_PATH, "|", "found" if TILE_PATH.exists() else "missing: the cells below need it")

started = time.perf_counter()
survey = load_parislille3d_data(TILE_PATH)
print(f"read in {time.perf_counter() - started:.1f} s")
print({key: tuple(value.shape) for key, value in survey.items()})

survey_extent = (survey["pos"].amax(0) - survey["pos"].amin(0)).tolist()
survey_area = survey_extent[0] * survey_extent[1]
print(f"{len(survey['pos']):,} points over {survey_extent[0]:.0f} x {survey_extent[1]:.0f} x "
      f"{survey_extent[2]:.0f} m ({survey_area / 1e4:.1f} ha), "
      f"{len(survey['pos']) / survey_area:.0f} points per square meter")

In [ ]:
DETAIL_HALF = 20.0  # meters: half a section, so the section is as wide as one block below

middle = 0.5 * (survey["pos"][:, 1].amin() + survey["pos"][:, 1].amax())
detail = ((survey["pos"][:, 1] - middle).abs() < DETAIL_HALF).nonzero().squeeze(1)

# reflectance is heavily skewed (median 9 of 255 here), so a square-root stretch is what spreads it
histogram = torch.bincount(survey["reflectance"][:, 0].long(), minlength=256)
high = int((histogram.cumsum(0) < 0.95 * len(survey["pos"])).sum())
reflectance = (survey["reflectance"][:, 0].float() / high).clamp(max=1.0).sqrt()
print(f"section: {len(detail):,} points | reflectance clipped at {high} of 255")

fig = plt.figure(figsize=(12, 5))
show_cloud(survey["pos"], survey["segment"], ax=fig.add_subplot(121, projection="3d"),
           title=f"the whole tile, {len(survey['pos']) / 1e6:.1f} M points, by the survey's own labels",
           vmin=0, vmax=len(PARISLILLE3D_CLASSES) - 1)
show_cloud(survey["pos"][detail], reflectance[detail], ax=fig.add_subplot(122, projection="3d"),
           title=f"one {2 * DETAIL_HALF:.0f} m section of it, by reflectance", view=STREET, cmap="viridis");

![The whole mobile-mapping tile colored by the survey's own class labels, next to one 40 m section of it colored by reflectance](../assets/tutorials/survey_tile.png)

That is a curved $310$ m corridor of street seen from almost straight above, drawn from $320$ thousand of its $21.4$ million points and colored by the survey's own labeling, which the model never sees and which we only use at the end to check the result. Ground, buildings and the street trees separate on color alone.

The right panel is one $40$ m section of the same tile colored by reflectance instead. The painted lines along the carriageway and the parked cars separate from the asphalt on that channel alone, which is why a surveyor never throws it away, and why the checkpoint below takes it as an input feature.

A tile of this size cannot be scattered point by point, so every panel on this page draws a fixed random subsample and says how many points it kept. The number on screen is never the size of the tile.

$21.4$ M points over $5.4$ ha at $394$ points per square meter. Voxelized at the checkpoint's $5$ cm working resolution, that is:

In [ ]:
whole = T.Voxelize(pos_key="pos", pos_reduce="grid", size=VOXEL)({"pos": survey["pos"].clone()})
print(f"{whole['pos'].shape[0]:,} occupied voxels at {VOXEL * 100:.0f} cm, "
      f"{len(survey['pos']) / whole['pos'].shape[0]:.1f} points each")

$7.16$ M voxels. At the $4.63$ GiB per million voxels this checkpoint costs, one pass over the whole
tile needs about $33$ GiB, which is past what a single GPU is likely to hold. That is the concrete version of the claim at the top: the tile is too big to run in one pass
at all.

### The checkpoint, and the frame it expects

`create_model(..., return_info=True)` hands back the exact preprocessing the weights were trained
with, and reading it is the first thing to do with an unfamiliar checkpoint.

In [ ]:
model, info = tp.create_model(
    "spvcnn-119gmacs.semantickitti.mit-han-lab",
    task="segmentation",
    pretrained=True,
    return_info=True,
)
model = model.to(device).eval()
print(model.num_classes, "classes:", list(info["weights"]["classes"])[:6], "...")
print(info["transform"])

The line that matters is `Cat(keys=('pos', 'intensity'), dst_key='x')`: this network takes raw
coordinates as **input features**, so it has learned an absolute frame. It was trained on
SemanticKITTI, where the sensor sits at the origin about $1.7$ m above the road. A survey tile is
delivered in project coordinates, hundreds or millions of meters from any origin, and feeding those
numbers in as features produces nonsense.

So the per-block pipeline has one more job than the registered transform: move each block into the
frame the checkpoint expects. Recenter $x$ and $y$ on the block, and put the local ground at
$z = -1.73$, estimated from a low quantile so a manhole or a scanner artifact does not drag it.

In [ ]:
SENSOR_HEIGHT = 1.73  # meters: where the training survey's sensor sat above the road


def to_sensor_frame(window):
    """Move a block into the frame the checkpoint was trained in: sensor at the origin, road below it."""
    data = dict(window)
    pos = data["pos"]
    center = 0.5 * (pos[:, :2].amin(0) + pos[:, :2].amax(0))
    ground = torch.quantile(pos[:, 2], 0.02)
    data["pos"] = torch.cat([pos[:, :2] - center, pos[:, 2:3] - ground - SENSOR_HEIGHT], dim=1)
    return data


prepare_block = T.Compose([
    to_sensor_frame,
    T.Cat(keys=["pos", "intensity"], dst_key="x", dim=1),
    T.Voxelize(pos_key="pos", pos_reduce="grid", keys=["x"], reduce=["first"], size=VOXEL,
               dst_inverse_key="inverse"),
])

That is the whole per-block pipeline: into the sensor frame, features assembled, voxelized with an
inverse map so the predictions can come back to the block's own points.

### Remapping class codes

The checkpoint predicts SemanticKITTI's 19 classes. The survey is annotated with its own 10-class
schema. Neither is a subset of the other, and the mapping is a decision you make once, in the open,
rather than a lookup table buried in a script.

In [ ]:
KITTI_TO_SURVEY = torch.zeros(19, dtype=torch.long)
for source, target in {
    0: 8, 1: 8, 2: 8, 3: 8, 4: 8,       # car, bicycle, motorcycle, truck, other-vehicle -> car
    5: 7, 6: 7, 7: 7,                   # person, bicyclist, motorcyclist -> pedestrian
    8: 1, 9: 1, 10: 1, 11: 1, 16: 1,    # road, parking, sidewalk, other-ground, terrain -> ground
    12: 2,                              # building
    13: 6,                              # fence -> barrier
    14: 9, 15: 9,                       # vegetation, trunk -> natural-vegetation
    17: 3, 18: 3,                       # pole, traffic-sign -> pole-road_sign-traffic_light
}.items():
    KITTI_TO_SURVEY[source] = target

unreachable = [
    name for i, name in enumerate(PARISLILLE3D_CLASSES)
    if i > 0 and i not in KITTI_TO_SURVEY.tolist()
]
print("survey classes no prediction can ever land on:", unreachable)

Two of the survey's classes, `bollard-small_pole` and `trash_can`, have no counterpart in the
checkpoint's schema, so their recall is structurally zero. Print that list before you run anything: it states what a borrowed checkpoint cannot deliver, and
it is the argument for fine-tuning on the survey's own schema rather than for a better mapping.

Mapping *many to one*, as here, is safe. Mapping one to many is not: if the survey distinguishes
`bollard` from `pole` and the model does not, no post-hoc rule recovers the difference, and pushing
one prediction into both codes silently invents data. For a delivered product, keep the model's own
class in an extra per-point field alongside the mapped survey code, so the mapping stays auditable.

### One guard the tiling forces on you

A tile never divides evenly into blocks, so the last row and column are slivers. A sparse
convolutional encoder that halves the resolution four times cannot run on a block spanning fewer
than $2^4 = 16$ voxels on some axis: the deepest stage comes back empty. Measured on this
checkpoint, $16$ voxels works and $15$ raises. The predictor guards it and abstains.

In [ ]:
MIN_SPAN = 16  # voxels: the encoder halves the resolution four times, so a thinner sliver has no deepest stage
skipped = [0]


def predict_block(block):
    """Run the checkpoint on one prepared block, abstaining on a sliver too thin for the encoder."""
    pos = block["pos"]
    if int((pos.amax(0) - pos.amin(0)).min()) + 1 < MIN_SPAN:
        skipped[0] += 1
        return pos.new_zeros((pos.shape[0], model.num_classes), dtype=torch.float32)
    return model(block["x"], pos, block["batch"])

An abstaining block returns a flat score vector, so its points come back with a uniform
distribution rather than a confident wrong answer. Count them: if the number is more than a handful,
the block size does not suit the tile.

Now run the tile. The dict holds the whole tile once, on the device; the inferer slices a block out
of it, hands the block through `prepare_block` to `predict_block`, and blends the results back.

In [ ]:
data = {
    "pos": survey["pos"].to(device),
    "intensity": (survey["reflectance"].float() / 255.0).to(device),
    "batch": torch.zeros(len(survey["pos"]), dtype=torch.long, device=device),
}

inferer = SlidingWindowInferer(
    block_size=40.0,
    overlap=0.25,
    mode="gaussian",
    dims=(0, 1),
    transform=prepare_block,
    inverse_key="inverse",
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
probabilities = inferer(data, predictor=predict_block)
torch.cuda.synchronize()
elapsed = time.perf_counter() - started

print(f"{len(survey['pos']):,} points in {elapsed:.1f} s "
      f"({len(survey['pos']) / elapsed / 1e6:.2f} M points/s), {skipped[0]} sliver(s) skipped")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print("output:", tuple(probabilities.shape), "| aligned:", probabilities.shape[0] == len(survey["pos"]))

$21.4$ M points in $6.1$ s, $3.5$ M points per second, one sliver skipped, $9.05$ GiB peak, and
the output is $(21402421, 19)$: one row per input point, in the input's order.

The tiling behind that can be inspected directly, with no model at all. The inferer slices
every per-point tensor of the input dict down to the block it is about to predict, so a predictor
handed the tile's own row numbers can record which block each point landed in.

In [ ]:
def block_map(overlap):
    """Walk the inferer's tiling over the tile: which block owns each point, and how many cover it."""
    owner = torch.zeros(len(survey["pos"]), dtype=torch.long)
    covered = torch.zeros(len(survey["pos"]), dtype=torch.long)
    visited = [0]

    def record(block):
        owner[block["index"]] = visited[0]
        covered[block["index"]] += 1
        visited[0] += 1
        return torch.zeros(len(block["index"]), 1)

    grid = {
        "pos": survey["pos"],
        "batch": torch.zeros(len(survey["pos"]), dtype=torch.long),
        "index": torch.arange(len(survey["pos"])),
    }
    SlidingWindowInferer(block_size=40.0, overlap=overlap, mode="constant", dims=(0, 1))(grid, predictor=record)
    return owner, covered, visited[0]


owner, _, partition = block_map(0.0)
_, covered, overlapping = block_map(0.25)
print(f"{partition} blocks as a strict partition, {overlapping} at 25% overlap")
for count, share in enumerate((torch.bincount(covered) / len(covered)).tolist()):
    if share > 0:
        print(f"  {share:6.1%} of the tile is predicted by {count} block(s)")

fig = plt.figure(figsize=(12, 5))
show_cloud(survey["pos"], owner, ax=fig.add_subplot(121, projection="3d"),
           title=f"{partition} blocks of 40 m, a strict partition")
show_cloud(survey["pos"], covered, ax=fig.add_subplot(122, projection="3d"), cmap="tab10", vmin=0, vmax=9,
           title=f"{overlapping} blocks at 25% overlap, by coverage");

![The tile colored by which sliding-window block predicts each point, next to the same tile colored by how many blocks cover each point at 25% overlap](../assets/tutorials/survey_blocks.png)

The left panel is the strict partition: $19$ blocks of $40$ m, each one a full-height column, and every point predicted exactly once. One block never appears there at all: it holds $5$ of the tile's $21\,402\,421$ points, a stray cluster $30$ cm across in a $40$ m cell, $2$ voxels on its narrowest axis against the $16$ the encoder needs. That is exactly the sliver the predictor above abstains on.

The right panel is the same grid at the $25\%$ overlap the run used. It needs $33$ blocks instead of $19$, and this is what the extra fourteen buy: $45.9\%$ of the tile is still predicted by one block, $42.2\%$ by two and $11.9\%$ by four. The bands and squares where that count rises fall exactly where a strict partition would leave a seam, and those points get a distance-weighted blend of every block that saw them rather than one block's opinion.

The tile carries its own labels, so we can ask how far a checkpoint trained on a different survey, a different sensor and a different schema actually gets.

In [ ]:
from torch_pointcloud.utils.metrics import confusion_matrix

predicted = KITTI_TO_SURVEY[probabilities.argmax(-1).cpu()]
truth = survey["segment"]
labeled = truth > 0  # class 0 is the survey's 'unclassified'
print(f"agreement with the survey's own labels: {(predicted[labeled] == truth[labeled]).float().mean():.1%}")

matrix = confusion_matrix(predicted, truth, len(PARISLILLE3D_CLASSES), ignore_index=0)
iou = matrix.diag() / (matrix.sum(0) + matrix.sum(1) - matrix.diag()).clamp_min(1)
for index, name in enumerate(PARISLILLE3D_CLASSES):
    present = int((truth == index).sum())
    if index > 0 and present > 0:
        print(f"  {name:34s} {present:>9,} points  IoU {iou[index]:.3f}")

$86.4\%$ of labeled points agree, with IoU $0.966$ on ground, $0.849$ on cars and $0.683$ on
buildings, against $0.000$ on the two classes the schema cannot express and $0.244$ on vegetation,
where the two surveys draw the tree-versus-facade boundary differently.

That measures the pipeline, not the model: nothing here was trained on this survey. A checkpoint fine-tuned on the survey's own schema would beat it on every line, and the
pipeline around it would not change at all.

Where that vegetation number comes from is easiest to see with the two labelings of the same points
side by side.

In [ ]:
fig = plt.figure(figsize=(12, 5))
shared = {"view": STREET, "vmin": 0, "vmax": len(PARISLILLE3D_CLASSES) - 1}
show_cloud(survey["pos"][detail], predicted[detail], ax=fig.add_subplot(121, projection="3d"), **shared,
           title="predicted, remapped onto the survey's schema")
show_cloud(survey["pos"][detail], survey["segment"][detail], ax=fig.add_subplot(122, projection="3d"), **shared,
           title="the survey's own labels over the same points");

![One 40 m section of the corridor at full point resolution, colored by the predicted class next to the survey's own labels for the same points](../assets/tutorials/survey_prediction.png)

Along the roofline, the transferred checkpoint calls the upper facades vegetation, which is the cross-survey error the vegetation IoU is reporting. The road, the sidewalks and the parked cars match the survey almost point for point. Both panels share one color scale, so a class keeps its color across them.

## Throughput and memory, measured

What one forward pass costs, and what the block size buys, are both measurable in a few lines on this tile.

In [ ]:
import numpy as np

PASS_BLOCKS = (10.0, 15.0, 20.0, 30.0, 40.0, 50.0)

middle_of_tile = survey["pos"][len(survey["pos"]) // 2, :2]
voxels, peaks = [], []
for block_size in PASS_BLOCKS:
    inside = ((survey["pos"][:, :2] - middle_of_tile).abs() < block_size / 2).all(1).nonzero().squeeze(1)
    window = prepare_block({"pos": survey["pos"][inside].clone(),
                            "intensity": survey["reflectance"][inside].float() / 255.0})
    x, grid = window["x"].to(device), window["pos"].to(device)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    with torch.inference_mode():
        model(x, grid, torch.zeros(len(grid), dtype=torch.long, device=device))
    torch.cuda.synchronize()
    voxels.append(len(grid) / 1e6)
    peaks.append((torch.cuda.max_memory_allocated() - before) / 2**30)
    print(f"{block_size:>5.0f} m block: {len(grid):>9,} voxels, peak {peaks[-1]:.3f} GiB")
    del x, grid, window
    torch.cuda.empty_cache()

slope = float(np.linalg.lstsq(np.array(voxels)[:, None], np.array(peaks), rcond=None)[0][0])
tile_voxels = whole["pos"].shape[0] / 1e6
card = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{slope:.2f} GiB per million voxels, so {tile_voxels:.2f} M voxels in one pass "
      f"would need {slope * tile_voxels:.0f} GiB, against {card:.0f} GiB available")

ax = plt.figure(figsize=(7, 4)).add_subplot()
ax.plot(voxels, peaks, "o-", label="measured, one forward pass")
ax.plot([voxels[-1], tile_voxels], [peaks[-1], slope * tile_voxels], "--", color="C0",
        label="the same slope, out to the whole tile")
ax.plot([0.0, tile_voxels], [card, card], "--", color="C2", label="available memory")
ax.set_xlabel("million voxels in one forward pass")
ax.set_ylabel("peak GPU memory (GiB)")
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False);

That is the argument for tiling, in numbers: peak memory is linear in the voxels of one pass at $4.63$ GiB per million, the fit is tight over the $88$ k to $1.04$ M voxels actually measured, and carrying the same slope out to the tile's $7.16$ M voxels reaches $33$ GiB, crossing the available memory long before that.

The block size is the knob that keeps a pass under that line, and it costs something. Five whole-tile runs put a number on both sides of the trade.

In [ ]:
SWEEP_BLOCKS = (15.0, 20.0, 30.0, 40.0, 50.0)

probabilities = probabilities.cpu()  # the 40 m run's scores are 1.51 GiB of the card on their own
torch.cuda.empty_cache()

throughput, peak_memory = [], []
for block_size in SWEEP_BLOCKS:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    scores = SlidingWindowInferer(
        block_size=block_size, overlap=0.25, mode="gaussian", dims=(0, 1),
        transform=prepare_block, inverse_key="inverse",
    )(data, predictor=predict_block)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    matches = KITTI_TO_SURVEY[scores.argmax(-1).cpu()][labeled] == truth[labeled]
    throughput.append(len(survey["pos"]) / elapsed / 1e6)
    peak_memory.append(torch.cuda.max_memory_allocated() / 2**30)
    print(f"{block_size:>5.0f} m: {elapsed:5.2f} s, {throughput[-1]:.2f} M points/s, "
          f"peak {peak_memory[-1]:5.2f} GiB, {matches.float().mean():.1%} agreement")
    del scores

ax = plt.figure(figsize=(7, 4)).add_subplot()
ax.plot(SWEEP_BLOCKS, throughput, "o-", label="throughput (M points/s)")
ax.plot(SWEEP_BLOCKS, peak_memory, "o-", label="peak GPU memory (GiB)")
ax.set_xlabel("block size (m)")
ax.set_ylabel("per whole-tile pass")
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False);

![Peak GPU memory against the voxels in one forward pass, next to the throughput and peak memory of one whole-tile pass by block size](../assets/tutorials/survey_throughput.png)

| block | blocks | seconds | M points/s | peak GiB | agreement |
| --- | --- | --- | --- | --- | --- |
| $15$ m | $150$ | $7.42$ | $2.88$ | $8.20$ | $87.9\%$ |
| $20$ m | $94$ | $6.49$ | $3.30$ | $8.20$ | $87.3\%$ |
| $30$ m | $48$ | $6.05$ | $3.54$ | $8.23$ | $86.6\%$ |
| $40$ m | $33$ | $5.80$ | $3.69$ | $9.05$ | $86.4\%$ |
| $50$ m | $24$ | $5.40$ | $3.96$ | $10.48$ | $85.4\%$ |

The left panel is the memory argument drawn: linear in the voxels of one pass, crossing the available memory at about $5$ M voxels, well short of the tile's $7.16$ M. The right panel is what the block size buys.

Throughput climbs while the blocks are small and the per-call cost dominates, then flattens: $15$ m to $30$ m buys $23\%$, and $30$ m to $50$ m buys $12\%$ more for $27\%$ more memory. The seconds depend on the machine and move by ten percent between runs; the block counts, the peaks and the agreement do not. Memory does move, and so does agreement, in the *opposite* direction from the usual intuition. Smaller blocks score better here because they put the model closer to the scenes it was trained on, which were single ego-centric scans a few tens of meters across. Blocks are a domain choice, not just a memory choice: $20$ m is the setting to ship for this pair of survey and checkpoint.

Below $8.2$ GiB the runs stop getting cheaper, and that floor is not the model. It is the bookkeeping: a $(21402421, 19)$ float32 score buffer is $1.51$ GiB on its own, and the inferer holds one of those plus the accumulator plus a slice of the tile's per-point tensors. Keeping the tile on the host and moving only each block to the device trades time for that floor.

In [ ]:
del data
torch.cuda.empty_cache()

host_tile = {
    "pos": survey["pos"],
    "intensity": survey["reflectance"].float() / 255.0,
    "batch": torch.zeros(len(survey["pos"]), dtype=torch.long),
}


def predict_block_on_host(block):
    pos = block["pos"].to(device)
    if int((pos.amax(0) - pos.amin(0)).min()) + 1 < MIN_SPAN:
        return torch.zeros(pos.shape[0], model.num_classes)
    return model(block["x"].to(device), pos, block["batch"].to(device)).cpu()


torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
host_probabilities = SlidingWindowInferer(
    block_size=40.0, overlap=0.25, mode="gaussian", dims=(0, 1),
    transform=prepare_block, inverse_key="inverse",
)(host_tile, predictor=predict_block_on_host)
elapsed = time.perf_counter() - started

print(f"tile on the host: {elapsed:.1f} s ({len(survey['pos']) / elapsed / 1e6:.2f} M points/s), "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
agree = (host_probabilities.argmax(-1) == probabilities.argmax(-1)).float().mean()
print(f"same label as the device-resident run on {agree:.4%} of points")

About $17$ s instead of $6$ s, and $5.08$ GiB instead of $9.05$ GiB: nearly three times slower for
$44\%$ less GPU memory. The labels match on $99.994\%$ of points, and the twelve hundred or so that
differ are decided by the order the blend accumulates in rather than by the model, which is what a
near-tie between two classes comes down to.

That is the knob to reach for when a tile grows faster than the card does, and it is what lets one
GPU process a tile of any size: only the blocks ever have to fit.

## Writing results back

Predictions are only useful attached to the points they belong to. Write the tile back with its
original coordinates, its original attributes and the new per-point fields.

In [ ]:
import numpy as np
from plyfile import PlyData, PlyElement

out_path = Path("Lille2_segmented.ply")
xyz = survey["pos"].numpy()
rows = np.empty(
    len(xyz),
    dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"), ("reflectance", "u1"),
           ("classification", "u1"), ("confidence", "f4")],
)
rows["x"], rows["y"], rows["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
rows["reflectance"] = survey["reflectance"].numpy()[:, 0]
rows["classification"] = predicted.numpy().astype(np.uint8)
rows["confidence"] = probabilities.amax(-1).numpy()

started = time.perf_counter()
PlyData([PlyElement.describe(rows, "vertex")], text=False).write(out_path.as_posix())
print(f"wrote {out_path.name}: {out_path.stat().st_size / 2**20:.0f} MiB "
      f"in {time.perf_counter() - started:.1f} s")

back = PlyData.read(out_path.as_posix())["vertex"]
print("rows read back:", len(back))
print("order preserved:", bool(np.array_equal(back["classification"], predicted.numpy().astype(np.uint8))))

$367$ MiB, written and read back in under a second, with the labels in the order they went out.
That last assertion is worth keeping in the pipeline rather than in a notebook: a per-point delivery
that is off by one row is worse than no delivery, and nothing downstream will notice.

Two notes for surveying workflows, where PLY is rarely the delivery format.

**LAS and LAZ.** The standard exchange format for airborne and mobile surveys carries a
`classification` field per point, an unsigned byte, with codes 0 to 255. ASPRS reserves the low
codes with fixed meanings ($2$ ground, $5$ high vegetation, $6$ building, and so on) and leaves
$64$ to $255$ for user definitions. Confidence, the model's own class before remapping, and any
other per-point output go in *extra bytes*: named, typed per-point dimensions declared in the header
that any LAS 1.4 reader will carry through. Writing those needs a LAS library, which
`torch-pointcloud` deliberately does not depend on; the tensors above are exactly what such a writer
wants, and the important part is that the row order is the file order.

**Class codes are a contract, not a convention.** A survey has a schema, ASPRS has a schema, and
your model has a label set, and all three differ. Keep the mapping in one place, next to the code
that applies it, the way `KITTI_TO_SURVEY` is above. Write both the mapped survey code and the
model's own class, and record which classes the mapping cannot reach: a delivery where
`bollard-small_pole` is silently absent looks the same as one where the model missed them.